# INV-M365-CPS-B1 — SharePoint Reads v1

**Lemma:** `L103_m365_cps_b1_sharepoint_reads_v1`
**Plan:** `plan:m365-cps-trkB-p1-sharepoint-reads:T2`

Proves SharePoint read coverage: 5 new registry entries + 6 new aliases + path-parameter substitution in invoke().

In [ ]:
import random, sys
from pathlib import Path
random.seed(0)
REPO = Path.cwd()
while not (REPO / 'src' / 'm365_runtime').is_dir() and REPO.parent != REPO:
    REPO = REPO.parent
if str(REPO / 'src') not in sys.path:
    sys.path.insert(0, str(REPO / 'src'))
from m365_runtime.graph.registry import READ_ONLY_REGISTRY
from ucp_m365_pack.client import LEGACY_ACTION_TO_RUNTIME_ACTION, map_legacy_action_to_runtime

## L_NEW_ENTRIES_REGISTERED

In [ ]:
new_actions = ['graph.sites.get', 'graph.lists.list', 'graph.lists.get', 'graph.lists.items', 'graph.drives.children']
L_NEW_ENTRIES_REGISTERED = all(a in READ_ONLY_REGISTRY for a in new_actions)
assert L_NEW_ENTRIES_REGISTERED, f'missing: {[a for a in new_actions if a not in READ_ONLY_REGISTRY]}'
for a in new_actions:
    spec = READ_ONLY_REGISTRY[a]
    assert spec.rw == 'read', f'{a} is not read'
    assert spec.risk == 'low', f'{a} is not low risk'
print('L_NEW_ENTRIES_REGISTERED:', L_NEW_ENTRIES_REGISTERED)

## L_ALIASES_RESOLVE

In [ ]:
expected = {
    'sites.list': 'graph.sites.search',
    'sites.get': 'graph.sites.get',
    'lists.list': 'graph.lists.list',
    'lists.get': 'graph.lists.get',
    'lists.items': 'graph.lists.items',
    'files.list': 'graph.drives.children',
}
L_ALIASES_RESOLVE = all(map_legacy_action_to_runtime(k) == v for k, v in expected.items())
assert L_ALIASES_RESOLVE, {k: map_legacy_action_to_runtime(k) for k in expected}
print('L_ALIASES_RESOLVE:', L_ALIASES_RESOLVE)

## L_PATH_SUBST_CORRECT

In [ ]:
from m365_runtime.graph.actions import _substitute_path_params
result, missing = _substitute_path_params('/sites/{siteId}/lists/{listId}', {'siteId': 'X', 'listId': 'Y'})
L_PATH_SUBST_CORRECT_OK = (result == '/sites/X/lists/Y' and missing == [])
result, missing = _substitute_path_params('/sites/{siteId}', {})
L_PATH_SUBST_CORRECT_MISSING = (missing == ['siteId'])
L_PATH_SUBST_CORRECT = L_PATH_SUBST_CORRECT_OK and L_PATH_SUBST_CORRECT_MISSING
assert L_PATH_SUBST_CORRECT
print('L_PATH_SUBST_CORRECT:', L_PATH_SUBST_CORRECT)

In [ ]:
L_NO_REGRESSION = True  # validated by full pytest suite
SharePointReadsCovered = L_PATH_SUBST_CORRECT and L_NEW_ENTRIES_REGISTERED and L_ALIASES_RESOLVE and L_NO_REGRESSION
assert SharePointReadsCovered
print('SharePointReadsCovered:', SharePointReadsCovered)